In [ ]:
import sys
import os

while not os.path.isdir("src") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from tradinglab.data_feed import DataFeed

In [ ]:
feed = DataFeed.from_dir(
    "data/egx",
    symbols=["ABUK"]
)

abuk_returns = feed.returns[:, 0].astype(np.float32)

print("Symbol:", feed.symbols[0])
print("Number of observations:", len(abuk_returns))
print("Returns shape:", abuk_returns.shape)

In [ ]:
print("First 10 returns:")
print(abuk_returns[:10])

print("\nMean return:", abuk_returns.mean())
print("Std return :", abuk_returns.std())
print("Min return :", abuk_returns.min())
print("Max return :", abuk_returns.max())

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(abuk_returns)

plt.title("ABUK Daily Returns")
plt.xlabel("Trading day")
plt.ylabel("Daily return")

plt.grid(alpha=0.3)
plt.show()

In [ ]:
LOOKBACK = 20

X = []
y = []

for i in range(LOOKBACK, len(abuk_returns)):
    
    X.append(
        abuk_returns[i - LOOKBACK:i]
    )
    
    y.append(
        abuk_returns[i]
    )

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
n = len(X)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

In [ ]:
train_mean = X_train.mean()
train_std = X_train.std()

print("Training mean:", train_mean)
print("Training std :", train_std)

In [ ]:
X_train_scaled = (X_train - train_mean) / train_std
X_val_scaled = (X_val - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

# Also normalize the target using training statistics
y_train_scaled = (y_train - train_mean) / train_std
y_val_scaled = (y_val - train_mean) / train_std
y_test_scaled = (y_test - train_mean) / train_std

In [ ]:
X_train_tensor = torch.tensor(
    X_train_scaled
).unsqueeze(-1)

X_val_tensor = torch.tensor(
    X_val_scaled
).unsqueeze(-1)

X_test_tensor = torch.tensor(
    X_test_scaled
).unsqueeze(-1)

y_train_tensor = torch.tensor(y_train_scaled)
y_val_tensor = torch.tensor(y_val_scaled)
y_test_tensor = torch.tensor(y_test_scaled)

print("X train:", X_train_tensor.shape)
print("y train:", y_train_tensor.shape)

In [ ]:
class ABUKLSTM(nn.Module):

    def __init__(
        self,
        input_size=1,
        hidden_size=64,
        num_layers=2,
        dropout=0.2
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        # Output from the final timestep
        last_output = output[:, -1, :]

        return self.head(last_output).squeeze(-1)

In [ ]:
model = ABUKLSTM()

print(model)

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=10
)

In [ ]:
EPOCHS = 500
PATIENCE = 30

train_history = []
val_history = []

best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(EPOCHS):

    # -----------------
    # Training
    # -----------------
    model.train()

    optimizer.zero_grad()

    train_pred = model(X_train_tensor)

    train_loss = criterion(
        train_pred,
        y_train_tensor
    )

    train_loss.backward()

    # Prevent exploding gradients
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    # -----------------
    # Validation
    # -----------------
    model.eval()

    with torch.no_grad():

        val_pred = model(X_val_tensor)

        val_loss = criterion(
            val_pred,
            y_val_tensor
        )

    train_loss_value = train_loss.item()
    val_loss_value = val_loss.item()

    train_history.append(train_loss_value)
    val_history.append(val_loss_value)

    scheduler.step(val_loss_value)

    # -----------------
    # Early stopping
    # -----------------
    if val_loss_value < best_val_loss:

        best_val_loss = val_loss_value

        best_state = {
            key: value.detach().clone()
            for key, value in model.state_dict().items()
        }

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

    if (epoch + 1) % 25 == 0:

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch+1:3d} | "
            f"train={train_loss_value:.6f} | "
            f"val={val_loss_value:.6f} | "
            f"lr={current_lr:.6f}"
        )

    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch+1}"
        )

        break

In [ ]:
model.load_state_dict(best_state)

print(
    f"Best validation loss: {best_val_loss:.6f}"
)

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(
    train_history,
    label="Train loss"
)

plt.plot(
    val_history,
    label="Validation loss"
)

plt.yscale("log")

plt.xlabel("Epoch")
plt.ylabel("MSE")

plt.title("ABUK LSTM — Training History")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
model.eval()

with torch.no_grad():

    test_predictions_scaled = (
        model(X_test_tensor)
        .cpu()
        .numpy()
    )

In [ ]:
test_predictions = (
    test_predictions_scaled * train_std
    + train_mean
)

actual_returns = y_test.copy()

print("Prediction shape:", test_predictions.shape)
print("Actual shape:", actual_returns.shape)

In [ ]:
test_mse = np.mean(
    (test_predictions - actual_returns) ** 2
)

test_rmse = np.sqrt(test_mse)

print(f"Test MSE : {test_mse:.8f}")
print(f"Test RMSE: {test_rmse:.8f}")

In [ ]:
SHOW_N = 150

plt.figure(figsize=(12, 5))

plt.plot(
    actual_returns[:SHOW_N],
    label="Actual return"
)

plt.plot(
    test_predictions[:SHOW_N],
    label="LSTM prediction",
    linestyle="--"
)

plt.axhline(
    0,
    linewidth=0.8
)

plt.title(
    "ABUK — LSTM Next-Day Return Prediction"
)

plt.xlabel("Test day")
plt.ylabel("Return")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
signals = (
    test_predictions > 0
).astype(float)

strategy_returns = (
    signals * actual_returns
)

print("Buy/Hold days:", int(signals.sum()))
print("Cash days:", int((signals == 0).sum()))

In [ ]:
lstm_total_return = (
    np.prod(1 + strategy_returns) - 1
)

buy_hold_return = (
    np.prod(1 + actual_returns) - 1
)

print(
    f"LSTM strategy return : {lstm_total_return:+.2%}"
)

print(
    f"Buy & hold return     : {buy_hold_return:+.2%}"
)

In [ ]:
def max_drawdown(returns):

    curve = np.cumprod(
        1 + returns
    )

    running_peak = np.maximum.accumulate(
        curve
    )

    drawdown = (
        running_peak - curve
    ) / running_peak

    return float(
        np.max(drawdown)
    )

In [ ]:
lstm_curve = np.cumprod(
    1 + strategy_returns
)

buy_hold_curve = np.cumprod(
    1 + actual_returns
)

plt.figure(figsize=(12, 5))

plt.plot(
    lstm_curve,
    label="LSTM strategy"
)

plt.plot(
    buy_hold_curve,
    label="Buy & Hold",
    linestyle="--"
)

plt.title(
    "ABUK — LSTM Strategy vs Buy & Hold"
)

plt.xlabel("Test day")
plt.ylabel("Portfolio value (normalized)")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
lstm_total_return = np.prod(1 + strategy_returns) - 1
lstm_max_dd = max_drawdown(strategy_returns)

buy_hold_return = np.prod(1 + actual_returns) - 1
buy_hold_max_dd = max_drawdown(actual_returns)

print(f"LSTM strategy return : {lstm_total_return:+.2%}")
print(f"Buy & hold return    : {buy_hold_return:+.2%}")
print(f"LSTM max drawdown    : {lstm_max_dd:.2%}")
print(f"Buy & hold drawdown  : {buy_hold_max_dd:.2%}")